# 05 Multimodal Inputs

So far, we have been providing _only_ text inputs to our Agents. In this workbook we'll illustrate how we can provide multi-modal inputs, such as images and audio to our agents. That should be fun, considering that in the next workbook we'll be building a fully functional _chef_ agent.

LLMs such as GPT models from OpenAI, Claude models from Anthropic and Gemini models from Google can ingest text or image or audio inputs and generate text or image or audio and even video outputs.

In this workbook we'll show you how to provide image and audio inputs to our Agents, which we'll be encoding into Base64 format - which encodes a binary (base 2) to 64 bits. This enables us to efficiently represent and transmit binary data, such as images and audio, on text-based communication channels.

In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

First let's see how we can _feed_ an image to our agent. We'll upload an image

In [3]:
from ipywidgets import FileUpload
from IPython.display import display

# upload single (multiple=False) PNG (accept=".png") files only!
uploader = FileUpload(accept=".png", multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [5]:
print(uploader.value[0])

{'name': 'moon.png', 'type': 'image/png', 'size': 358916, 'content': <memory at 0x000001F22978D0C0>, 'last_modified': datetime.datetime(2026, 4, 14, 14, 8, 35, 140000, tzinfo=datetime.timezone.utc)}


In [6]:
# let's encode the image using base64 encoding
import base64

uploaded_file = uploader.value[0]
encoded_image = base64.b64encode(bytes(uploaded_file["content"])).decode("utf-8")

In [ ]:
# now let's pass a multi-modal question to our agent
from langchain_core.messages import HumanMessage

multimodal_question = HumanMessage(
    {"type": "text", "text": "Tell me about this image"},
    {"type": "image", "base64": encoded_image, "mime-type": "image/png"},
)

response = agent.invoke({"messages": [multimodal_question]})
print(response["messages"][-1].content)

In [2]:
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-5-nano",
    # here I am giving some examples to guide the agent on how to respond
    system_prompt="""
    You are an expert on historical events, places of interest and monuments.
    Answer the user's question to the best of your ability. Keep your resoponses
    crisp and to the point - don't provide too much information, just what's asked for.

    Examples:
    Q: What is the capital of France?
    A: Paris.
    Q: Name some historical monuments in Rome.
    A: The Colosseum, The Pantheon, The Roman Forum.
    """,
)

print("-" * 90)
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the capital of USA?"}]}
)
print(response["messages"][-1].content)

print("-" * 90)

# ask a follow-up question - does the agent respond correctly?
response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Name some historical monuments in that city"}
        ]
    }
)
print(response["messages"][-1].content)

------------------------------------------------------------------------------------------
Washington, D.C.
------------------------------------------------------------------------------------------
Which city are you referring to? I can list its historical monuments. For example, in Rome: Colosseum, Pantheon, Roman Forum.


As expected, that was a disaster!

Clearly, the agent's interaction is stateless, meaning each question & response (or inference) has no clue of any other question & response - before & after.

With a LangChain agent we track messages within something called a _State_. You can think of it as the memory of the agent. The problem is that the _State_ **isn't begin saved** from one run (Q&R) to another. So, in effect our Agent's memory is being wiped clean between each interaction. We somehow need to save our state such that the Agent can remember previous questions & responses (all System, User and AI messages exchanged so far).

We do that by using something called a _Check Pointer_, which saves a snapshot of the state at the end of each run and groups it with other runs with the same thread ID. The _Check Pointer_ we use is a class called `InMemorySaver`, which is imported from `langgraph.checkpoint.memory` package - it's `langgraph` not `langchain`!

<div align="center">
<img src="images/saving_state.png" width="450" heigh="500" alt="Saving State"/>
</div>

Now let's enable our agent with memory & see the difference in its responses.

In [6]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent2 = create_agent(
    model="openai:gpt-5-nano",
    # here I am giving some examples to guide the agent on how to respond
    system_prompt="""
    You are an expert on historical events, places of interest and monuments.
    Answer the user's question to the best of your ability. Keep your resoponses
    crisp and to the point - don't provide too much information, just what's asked for.

    Examples:
    Q: What is the capital of France?
    A: Paris.
    Q: Name some historical monuments in Rome.
    A: The Colosseum, The Pantheon, The Roman Forum.
    """,
    # here we add the memory
    checkpointer=InMemorySaver(),
)

# since we are using checkpoints, we need to provide a common configuration
# so the agents can group all previous interactions.
config = {"configurable": {"thread_id": "1024"}}  # any number is ok!

print("-" * 90)
response = agent2.invoke(
    {"messages": [{"role": "user", "content": "What is the capital of USA?"}]},
    config=config,
)
print(response["messages"][-1].content)

print("-" * 90)

# ask a follow-up question - does the agent respond correctly?
response = agent2.invoke(
    {
        "messages": [
            {"role": "user", "content": "Name some historical monuments in that city"}
        ]
    },
    config=config,
)
print(response["messages"][-1].content)

------------------------------------------------------------------------------------------
Washington, D.C.
------------------------------------------------------------------------------------------
- Washington Monument
- Lincoln Memorial
- Jefferson Memorial
- World War II Memorial
- Vietnam Veterans Memorial
- Korean War Veterans Memorial
- Franklin Delano Roosevelt Memorial
- Martin Luther King Jr. Memorial


Fantastic! Now our agent is able to link the next question to previous questions & responses!

To see the complete list of messages, let's examine `response['messages']`

In [ ]:
# just showing the content of each message agent has "seen so far"=
[response["messages"][i].content for i in range(len(response["messages"]))]

['What is the capital of USA?',
 'Washington, D.C.',
 'Name some historical monuments in that city',
 '- Washington Monument\n- Lincoln Memorial\n- Jefferson Memorial\n- World War II Memorial\n- Vietnam Veterans Memorial\n- Korean War Veterans Memorial\n- Franklin Delano Roosevelt Memorial\n- Martin Luther King Jr. Memorial']

So you can see that the agent has _retained_ all pevious messages, which it can now _refer to_ to generate the response to the latest question. This list of messages is _fed_ internally by the agent as the _context_ to the LLM/model, which can then generate an appropriate response.

So now you have learn't how to create a [almost] complete Agent.
- You created a basic agent
- You provided it with tools
- You equipped it with memory to remeber past messages!

Enough to create a fully functional chat-bot!